<img src="https://raw.githubusercontent.com/carubbi/MQ/main/notebooks/assets/imgs/unifor-logo.png" width="400">
<br>
<b>
<font size="6" face="arial" color="blue">
    Graduação em Ciência da Computação
</font>
</b>
<br>
<b>
<font size="4" face="arial">
    Disciplina: Métodos Quantitativos em Computação
</font>
</b>

**Orientador: Prof. Me. Ricardo Carubbi** <br>
*Docente da Graduação e Pós-Graduação em Ciência de Dados e Inteligência Artificial*<br>
*Laboratório de Ciência de Dados e Inteligência Artificial*<br>
*Universidade de Fortaleza*<br>

# **Aula 10 — Associação entre Variáveis Quantitativas**

## 1. Objetivos de aprendizagem

Ao final da aula, o aluno será capaz de:

- interpretar direção, forma e agrupamentos em um diagrama de dispersão.
- calcular covariância amostral e correlação de Pearson usando os mesmos pares completos.
- interpretar o coeficiente junto ao gráfico, considerando grupos e observações discrepantes.
- reconhecer a inversão da associação no paradoxo de Simpson e distinguir associação de causalidade.

## 2. Agenda

1. Diagrama de dispersão — 20 min
2. Covariância e correlação — 30 min
3. Limites da interpretação — 20 min
4. Acompanhamento da AP1 — 30 min

**Demonstrações prontas:** carregamento, gráficos adicionais e comparações por espécie e sexo. O cálculo manual e a interpretação serão desenvolvidos com a turma.

## 3. Diagrama de dispersão

Pinguins com nadadeiras mais compridas tendem a apresentar maior massa corporal?

No **diagrama de dispersão**, cada ponto representa duas medidas do mesmo indivíduo. Observe direção, forma, concentração e grupos antes de resumir a relação por um número. No Q–Q da Aula 9, cada ponto comparava quantis, e não medidas do mesmo indivíduo. (BARBETTA; REIS; BORNIA, 2010, pp. 317–318)

Usaremos a base preparada na Aula 5. **Pares completos** são registros com as duas medidas preenchidas.

In [ ]:
# Importar as bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar os gráficos
sns.set_theme(style='whitegrid')
especies = ['Adelie', 'Chinstrap', 'Gentoo']
cores = {'Adelie': 'darkorange', 'Chinstrap': 'purple', 'Gentoo': '#008B8B'}
marcadores = {'Adelie': 'o', 'Chinstrap': 's', 'Gentoo': '^'}

In [ ]:
# Carregar a base processada do GitHub
url = 'https://raw.githubusercontent.com/carubbi/MQ/main/data/processed/penguins.csv'
df = pd.read_csv(url)
df.head()

In [ ]:
# Selecionar os mesmos pares para todos os cálculos
pares = df.dropna(subset=['FLIPPER_LENGTH', 'BODY_MASS'])
print('Registros na base:', len(df))
print('Pares completos:', len(pares))

In [ ]:
# Comparar nadadeira e massa no conjunto
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=pares, x='FLIPPER_LENGTH', y='BODY_MASS',
                color='black', marker='o', ax=ax)
ax.set(xlabel='Comprimento da nadadeira (mm)', ylabel='Massa corporal (g)')
plt.tight_layout()
plt.show()

**Figura 1 — Comprimento da nadadeira e massa corporal.** Nos 342 pares completos, nadadeiras mais compridas tendem a acompanhar massas maiores. A nuvem de pontos apresenta uma tendência crescente.

Fonte: CARUBBI, 2026, com dados de HORST; HILL; GORMAN, 2020.

## 4. Covariância e correlação

A covariância resume como duas medidas variam juntas. A correlação de Pearson padroniza esse resultado para permitir uma interpretação sem unidades.

### 4.1 Covariância amostral

Um **desvio** é a diferença entre um valor e a média. Se os dois desvios têm o mesmo sinal, seu produto é positivo. Se têm sinais opostos, o produto é negativo.

A **covariância amostral** é a soma desses produtos dividida por $n-1$:

$$s_{xy}=\frac{\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})}{n-1}.$$

- $x_i$ e $y_i$: medidas do mesmo indivíduo.
- $\bar{x}$ e $\bar{y}$: médias das medidas nos mesmos pares completos.
- $n$: número de pares completos, maior que 1.
- $s_{xy}$: covariância amostral.

Covariância positiva indica um saldo positivo dos produtos. Covariância negativa indica um saldo negativo. Seu valor depende das unidades das duas medidas. (PINHEIRO et al., 2009, p. 67)

Usaremos três pares reais apenas para acompanhar o cálculo. Esse recorte não representa a associação da base inteira.

A covariância pode ser zero quando os produtos positivos e negativos se compensam. Isso não exclui uma relação não linear. (MORETTIN; BUSSAB, 2010, pp. 100–101)

Mantenha a precisão durante os cálculos. Arredonde apenas os resultados apresentados. (BARBETTA; REIS; BORNIA, 2010, p. 322)

In [ ]:
# Selecionar três pares reais para o cálculo manual
amostra = pares.iloc[[0, 100, 250]]
x = amostra['FLIPPER_LENGTH']
y = amostra['BODY_MASS']
amostra[['ID', 'FLIPPER_LENGTH', 'BODY_MASS']]

In [ ]:
# Atividade: Calcule n, mx e my com soma e divisão, usando x e y.

O primeiro par está abaixo das duas médias, e o terceiro está acima delas. Seus produtos de desvios são positivos. No segundo par, a nadadeira é igual à média, tornando o produto zero. O saldo positivo indica covariância positiva.

In [ ]:
# Atividade: Calcule dx e dy e apresente o produto dos desvios de cada par.

In [ ]:
# Exibir os desvios da massa
print('Desvios da massa (g):')
print(dy)

In [ ]:
# Exibir os produtos dos desvios
print('Produtos dos desvios (mm x g):')
print(dx * dy)

In [ ]:
# Calcular a covariância pela definição
def covariancia(x: pd.Series, y: pd.Series) -> float:
    """Calcula a covariância amostral de pares completos."""
    if len(x) < 2 or not x.index.equals(y.index):
        raise ValueError('Use pelo menos dois pares com os mesmos índices.')
    if x.isna().any() or y.isna().any():
        raise ValueError('Selecione os pares completos antes do cálculo.')
    n = len(x)
    mx = sum(x) / n
    my = sum(y) / n
    soma = 0.0
    for xi, yi in zip(x, y):
        soma += (xi - mx) * (yi - my)
    return soma / (n - 1)

In [ ]:
# Conferir a covariância dos três pares
print('Pela função (mm x g):', covariancia(x, y))
print('Pelo pandas (mm x g):', x.cov(y))

A covariância dos três pares é **16.500 mm × g**. O resultado positivo acompanha a tendência crescente do pequeno conjunto.

In [ ]:
# Examinar o efeito da unidade na base completa
x_base = pares['FLIPPER_LENGTH']
y_base = pares['BODY_MASS']
print('Covariância em mm x g:', x_base.cov(y_base))
print('Covariância em mm x kg:', x_base.cov(y_base / 1000))

Ao converter gramas para quilogramas, a covariância é dividida por 1.000. A relação observada permanece a mesma, mas o número muda com a unidade.

### 4.2 Correlação linear de Pearson

O **coeficiente de correlação de Pearson** divide a covariância pelo produto dos desvios-padrão amostrais:

$$r=\frac{s_{xy}}{s_xs_y}, \qquad -1\leq r\leq 1.$$

- $s_x$ e $s_y$: desvios-padrão amostrais das duas medidas.
- $s_{xy}$: covariância amostral.
- $r$: coeficiente de correlação linear, sem unidade.

Use os mesmos pares completos em todos os cálculos. As duas variáveis devem apresentar desvio-padrão positivo.

- $r>0$: associação linear positiva.
- $r<0$: associação linear negativa.
- Quanto mais próximo de 1 estiver $|r|$, mais intenso é o alinhamento linear.
- $r=1$ ou $r=-1$: alinhamento linear perfeito.
- $r$ próximo de zero: pouca associação linear.

Interprete a magnitude junto ao gráfico. O coeficiente não é um percentual de associação nem de variabilidade explicada. (BARBETTA; REIS; BORNIA, 2010, pp. 319–322)

Retomando o desvio-padrão da Aula 7:

$$s_x=\sqrt{\frac{\sum_{i=1}^{n}(x_i-\bar{x})^2}{n-1}},\qquad
s_y=\sqrt{\frac{\sum_{i=1}^{n}(y_i-\bar{y})^2}{n-1}}.$$

**Padronizar** é dividir o desvio em relação à média pelo desvio-padrão. O resultado informa quantos desvios-padrão o valor está acima ou abaixo da média. Assim, medidas com unidades diferentes ficam em uma escala comparável. (BARBETTA; REIS; BORNIA, 2010, p. 319)

$$z_x=\frac{x_i-\bar{x}}{s_x},\qquad z_y=\frac{y_i-\bar{y}}{s_y}.$$

- $z_x$ e $z_y$: valores padronizados das duas medidas de um par.


In [ ]:
# Padronizar uma série de medidas
def padronizar(valores: pd.Series) -> pd.Series:
    """Expressa os desvios em unidades de desvio-padrão amostral."""
    if valores.count() < 2 or valores.std() == 0:
        raise ValueError('Use pelo menos dois valores válidos e com variabilidade.')
    return (valores - valores.mean()) / valores.std()

In [ ]:
# Padronizar as medidas e observar o primeiro par
zx = padronizar(x)
zy = padronizar(y)
print('Nadadeira padronizada:', round(zx.iloc[0], 3))
print('Massa padronizada:', round(zy.iloc[0], 3))

No primeiro par, a nadadeira está **1 desvio-padrão abaixo** da média, e a massa está aproximadamente **1,084 desvios-padrão abaixo**. Os dois valores padronizados são negativos e não têm unidade.

A covariância de uma variável com ela mesma é sua variância amostral. Por isso, a função usa `covariancia(x, x)` e `covariancia(y, y)` para obter os desvios-padrão.

In [ ]:
# Calcular Pearson pela definição
def correlacao(x: pd.Series, y: pd.Series) -> float:
    """Padroniza a covariância pelos desvios-padrão amostrais."""
    cov = covariancia(x, y)
    sx = covariancia(x, x) ** 0.5
    sy = covariancia(y, y) ** 0.5
    if sx == 0 or sy == 0:
        raise ValueError('As duas variáveis devem apresentar variabilidade.')
    return cov / (sx * sy)

In [ ]:
# Conferir Pearson nos três pares
print('Pela função:', correlacao(x, y))
print('Pelo pandas:', x.corr(y))

Nos três pares, $r\approx 0,985$, indicando alinhamento linear positivo intenso. Esse resultado descreve somente o pequeno recorte.

A correlação é **simétrica**: $r_{xy}=r_{yx}$. Trocar a ordem das variáveis preserva o coeficiente, sem estabelecer uma direção causal.

In [ ]:
# Conferir a simetria da correlação
print('x com y:', correlacao(x, y))
print('y com x:', correlacao(y, x))

In [ ]:
# Atividade: Calcule a correlação entre x_base e y_base e confira o resultado com massa em kg.

Na Figura 1, a correlação geral é aproximadamente **0,871**, indicando associação linear positiva intensa. A conversão da massa para quilogramas preserva $r$.

In [ ]:
# Comparar nadadeira e comprimento do bico
bico = df.dropna(subset=['FLIPPER_LENGTH', 'CULMEN_LENGTH'])
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=bico, x='FLIPPER_LENGTH', y='CULMEN_LENGTH',
                color='black', marker='o', ax=ax)
ax.set(xlabel='Comprimento da nadadeira (mm)', ylabel='Comprimento do bico (mm)')
plt.tight_layout()
plt.show()

**Figura 2 — Comprimento da nadadeira e do bico.** A associação geral é positiva, com r ≈ 0,656. O alinhamento linear é menos intenso que na relação entre nadadeira e massa. Fonte: CARUBBI, 2026, com dados de HORST; HILL; GORMAN, 2020.

In [ ]:
# Conferir a correlação geral
comp_nadadeira = bico['FLIPPER_LENGTH']
comp_bico = bico['CULMEN_LENGTH']
print('Geral | n =', len(bico), '| r =', round(comp_nadadeira.corr(comp_bico), 3))

### 4.3 Matriz de correlação

A **matriz de correlação** reúne os coeficientes de Pearson de todos os pares de variáveis selecionadas. Cada linha e coluna identifica uma medida.

- A diagonal vale 1, pois compara cada variável com ela mesma.
- A matriz é simétrica, pois trocar as variáveis preserva a correlação.
- Cada célula fora da diagonal descreve a associação linear entre duas medidas.

Usaremos os registros com as quatro medidas preenchidas para manter o mesmo conjunto em todas as comparações. (BARBETTA; REIS; BORNIA, 2010, p. 321)

In [ ]:
# Calcular a matriz de correlação das medidas
cols = ['CULMEN_LENGTH', 'CULMEN_DEPTH', 'FLIPPER_LENGTH', 'BODY_MASS']
medidas = df[cols].dropna()
print('Registros completos:', len(medidas))
medidas.corr().round(3)

O **mapa de calor (heatmap)** representa os coeficientes por cores. Azul indica correlação negativa e vermelho, positiva. Tons próximos do branco indicam valores próximos de zero.

In [ ]:
# Representar a matriz de correlação por cores
rotulos = ['Comprimento do bico', 'Profundidade do bico', 'Nadadeira', 'Massa corporal']
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(medidas.corr(), annot=True, fmt='.3f', cmap='RdBu_r',
            vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
            xticklabels=rotulos, yticklabels=rotulos,
            cbar_kws={'label': 'Correlação de Pearson (r)'}, ax=ax)
ax.set_xticklabels(rotulos, rotation=30, ha='right')
ax.set_yticklabels(rotulos, rotation=0)
plt.tight_layout()
plt.show()

**Figura 3 — Mapa de calor das correlações entre medidas.** Fora da diagonal, nadadeira e massa apresentam a maior correlação positiva (0,871). Comprimento e profundidade do bico apresentam correlação negativa (−0,235). As cores resumem a matriz e não substituem a análise dos diagramas de dispersão.

Fonte: CARUBBI, 2026, com dados de HORST; HILL; GORMAN, 2020.

A maior correlação positiva entre medidas diferentes ocorre entre nadadeira e massa (**0,871**). Comprimento e profundidade do bico apresentam correlação negativa (**−0,235**). Esses valores descrevem o conjunto completo e devem ser interpretados junto aos gráficos.

## 5. Limites da interpretação

O mesmo coeficiente pode acompanhar padrões gráficos diferentes. Examine a forma da relação, os pontos e os grupos antes de concluir.

### 5.1 Linearidade e valores discrepantes

Pearson resume o alinhamento **linear**. Um coeficiente próximo de zero não exclui uma relação curva. Um ponto afastado do padrão dos demais pares pode aumentar ou diminuir a correlação, conforme sua posição. (PINHEIRO et al., 2009, p. 71)

No exemplo artificial abaixo, a segunda variável é o quadrado da primeira. A relação é curva, embora a correlação linear seja zero.

In [ ]:
# Criar uma relação curva com dados artificiais
u = pd.Series([-3, -2, -1, 0, 1, 2, 3])
v = u ** 2
print('Correlação:', round(u.corr(v), 3))

In [ ]:
# Mostrar a relação entre as variáveis artificiais
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(u, v, s=70)
ax.set(xlabel='u', ylabel='v = u²')
plt.tight_layout()
plt.show()

**Figura 4 — Relação não linear em dados artificiais.** Cada valor de v é determinado por u². Apesar dessa relação, r = 0 porque o padrão em U não tem direção linear predominante. Os produtos de desvios positivos e negativos se compensam.

Fonte: CARUBBI, 2026.

Agora, vamos alterar a massa de um dos três pares reais em uma cópia didática para observar a sensibilidade da correlação.

In [ ]:
# Alterar uma massa somente na cópia didática
y_alt = y.copy()
y_alt.iloc[2] = 2500
r_original = x.corr(y)
r_alterado = x.corr(y_alt)  
print(f'Correlação original: {r_original:.3f}')
print(f'Correlação após alteração: {r_alterado:.3f}')

In [ ]:
# Mostrar a substituição de uma massa no pequeno conjunto
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x.iloc[:2], y.iloc[:2], s=70, label='Pares inalterados')
ax.scatter(x.iloc[2], y.iloc[2], color='gray', s=80, label='Ponto original (5.250 g)')
ax.scatter(x.iloc[2], y_alt.iloc[2], color='red', marker='X', s=100,
           label='Ponto artificial (2.500 g)')
ax.annotate('', xy=(x.iloc[2], y_alt.iloc[2]), xytext=(x.iloc[2], y.iloc[2]),
            arrowprops={'arrowstyle': '->', 'color': 'red', 'linestyle': '--'})
ax.set(xlabel='Comprimento da nadadeira (mm)', ylabel='Massa corporal (g)')
ax.legend(
    title=f'r original = {r_original:.3f}\nr após alteração = {r_alterado:.3f}', 
    loc='upper left'
)
plt.tight_layout()
plt.show()

**Figura 5 — Sensibilidade da correlação à alteração de uma observação.** A substituição artificial de 5.250 g por 2.500 g no terceiro par muda r de 0,985 para −0,560. A inversão da direção da associação neste pequeno conjunto evidencia a sensibilidade da correlação a valores discrepantes. A base original permanece preservada.

Fonte: CARUBBI, 2026, com dados de HORST; HILL; GORMAN, 2020.

**Correlação não demonstra causalidade.** A associação entre duas medidas não prova que alterar uma delas provoque mudança na outra. (BARBETTA; REIS; BORNIA, 2010, pp. 317–318)

### 5.2 Paradoxo de Simpson

Até aqui, examinamos as relações no conjunto. Agora, calcule a correlação geral entre comprimento e profundidade do bico. Depois, identifique as espécies no gráfico e compare as correlações dentro de cada uma.

In [ ]:
# Calcular Pearson antes de identificar as espécies
bico = df.dropna(subset=['CULMEN_LENGTH', 'CULMEN_DEPTH'])
print('Geral | n =', len(bico), '| r =', round(bico['CULMEN_LENGTH'].corr(bico['CULMEN_DEPTH']), 3))

In [ ]:
# Comparar as medidas do bico por espécie
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=bico, x='CULMEN_LENGTH', y='CULMEN_DEPTH',
                hue='SPECIES', style='SPECIES', hue_order=especies,
                style_order=especies, markers=marcadores, palette=cores, ax=ax)
ax.set(xlabel='Comprimento do bico (mm)', ylabel='Profundidade do bico (mm)')
ax.legend(title='Espécie')
plt.tight_layout()
plt.show()

In [ ]:
# Calcular Pearson dentro de cada espécie
for especie in especies:
    grupo = bico[bico['SPECIES'] == especie]
    r = grupo['CULMEN_LENGTH'].corr(grupo['CULMEN_DEPTH'])
    print(especie, '| n =', len(grupo), '| r =', round(r, 3))

**Figura 6 — Associação agregada e por espécie.** A correlação é negativa no conjunto (r ≈ −0,235) e positiva em Adelie (0,391), Chinstrap (0,654) e Gentoo (0,643). Ao reunir as espécies, as diferenças entre os grupos produzem uma tendência oposta à observada dentro de cada um.

Fonte: CARUBBI, 2026, com dados de HORST; HILL; GORMAN, 2020.

O **paradoxo de Simpson** ocorre quando uma associação observada nos grupos se inverte ao reuni-los. Aqui, o sinal geral é oposto ao sinal dentro de cada espécie. (MONTGOMERY; RUNGER, 2018, p. 548)

Retome agora as relações das Figuras 1 e 2 e calcule suas correlações dentro de cada espécie.

In [ ]:
# Comparar a correlação dentro de cada espécie
for especie in especies:
    grupo = pares[pares['SPECIES'] == especie]
    r = grupo['FLIPPER_LENGTH'].corr(grupo['BODY_MASS'])
    print(especie, '| n =', len(grupo), '| r =', round(r, 3))

In [ ]:
# Retomar nadadeira e comprimento do bico dentro das espécies
for especie in especies:
    grupo = df[df['SPECIES'] == especie].dropna(subset=['FLIPPER_LENGTH', 'CULMEN_LENGTH'])
    r = grupo['FLIPPER_LENGTH'].corr(grupo['CULMEN_LENGTH'])
    print(especie, '| n =', len(grupo), '| r =', round(r, 3))

Para nadadeira e massa, r é **0,468** em Adelie, **0,642** em Chinstrap e **0,703** em Gentoo, frente a **0,871** no conjunto. Para nadadeira e comprimento do bico, os valores são **0,326**, **0,472** e **0,661**, frente a **0,656** no conjunto. Essas comparações mostram mudanças de intensidade, sem a inversão do sinal observada na Figura 6. A composição dos grupos influencia a correlação geral. Nenhuma dessas associações demonstra causalidade.

### 5.3 Associação por espécie e sexo

Separar os painéis por espécie e identificar o sexo ajuda a observar agrupamentos que o gráfico geral pode ocultar. Nesta análise, também precisamos do sexo preenchido.

In [ ]:
# Selecionar os registros válidos para os painéis
sexo = pares.dropna(subset=['SPECIES', 'SEX'])
print('Pares sem exigir sexo:', len(pares))
print('Pares com sexo preenchido:', len(sexo))
print(sexo.groupby('SPECIES').size())

In [ ]:
# Comparar nadadeira e massa por espécie e sexo
g = sns.relplot(data=sexo, x='FLIPPER_LENGTH', y='BODY_MASS',
                col='SPECIES', col_order=especies, hue='SEX', style='SEX',
                hue_order=['female', 'male'], style_order=['female', 'male'],
                palette={'female': '#9467bd', 'male': '#2878b5'},
                kind='scatter', height=3.6, aspect=1)
g.set_axis_labels('Comprimento da nadadeira (mm)', 'Massa corporal (g)')
g.set_titles('{col_name}')
g.legend.set_title('Sexo')
for texto in g.legend.get_texts():
    texto.set_text({'female': 'Fêmea', 'male': 'Macho'}[texto.get_text()])
plt.show()

**Figura 7 — Nadadeira e massa por espécie e sexo.** Nos 333 pares com sexo preenchido, os machos tendem a ocupar massas e comprimentos de nadadeira maiores, com sobreposição entre os sexos. Os painéis mostram que a composição dos grupos também deve ser considerada dentro de cada espécie.

Fonte: CARUBBI, 2026, com dados de HORST; HILL; GORMAN, 2020.

## 6. Acompanhamento da AP1

Nos 30 minutos finais, revise com seu grupo a análise univariada do Ames Housing:

1. Medidas descritivas e suas unidades.
2. Histogramas e gráficos de barras de $B$, conforme a especificação do trabalho.
3. Quartis, intervalo interquartil, cercas de Tukey e boxplots.
4. Decisão preliminar justificada sobre valores discrepantes.

Registre as interpretações, dúvidas e ajustes necessários no notebook do trabalho. Consulte a [especificação da AP1](https://github.com/carubbi/MQ/blob/main/trabalhos/ap1.md).

## 7. Estudo e exercícios

### 7.1 Materiais didáticos

- BARBETTA; REIS; BORNIA (2010), seção 11.1, pp. 317–318: dispersão e associação.
- BARBETTA; REIS; BORNIA (2010), seção 11.2, pp. 319–322, até a correlação amostral: padronização e Pearson.
- PINHEIRO et al. (2009), seção 2.2, pp. 66–71: gráfico, covariância, correlação e limites. Priorize esses tópicos na leitura.
- MORETTIN; BUSSAB (2010), seção 4.5, pp. 100–102: leitura complementar. Na aula, adotamos covariância e desvios-padrão amostrais com denominador $n-1$.

### 7.2 Exercícios indicados

- PINHEIRO et al. (2009), exercício 2.7_P, p. 85, somente itens c e d. Justifique as respostas pela interpretação e pelo intervalo do coeficiente.

**Atividade com a base da aula:** selecione outros três pares completos com `pares.iloc[[10, 50, 200]]`. Calcule covariância e correlação com as funções manuais e confira com pandas. Interprete o sinal e a intensidade da associação linear. Explique por que o resultado desse recorte não descreve necessariamente a base inteira.

## 8. Referências

1. BARBETTA, Pedro Alberto; REIS, Marcelo Menezes; BORNIA, Antonio Cezar. *Estatística para cursos de engenharia e informática*. 3. ed. São Paulo: Atlas, 2010.
2. PINHEIRO, João Ismael D. et al. *Estatística básica: a arte de trabalhar com dados*. Rio de Janeiro: Elsevier, 2009.
3. MORETTIN, Pedro A.; BUSSAB, Wilton O. *Estatística básica*. 6. ed. São Paulo: Saraiva, 2010.
4. HORST, Allison Marie; HILL, Alison Presmanes; GORMAN, Kristen B. *palmerpenguins: Palmer Archipelago (Antarctica) penguin data*. Zenodo, 2020. DOI: [10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218).
5. MONTGOMERY, Douglas C.; RUNGER, George C. *Applied Statistics and Probability for Engineers*. 7. ed. Hoboken: Wiley, 2018.